In [2]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DFExample") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/26 14:27:07 WARN Utils: Your hostname, aswin-Inspiron, resolves to a loopback address: 127.0.1.1; using 192.168.127.189 instead (on interface wlp1s0)
26/02/26 14:27:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 14:27:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [1]:
import os
print(os.environ.get("JAVA_HOME"))

/usr/lib/jvm/java-17-openjdk-amd64


In [23]:
df = spark.read.csv("transactions.csv", header=True, inferSchema=True)
df.show()

+--------------+-------+------------+---------------+------+-------------------+
|transaction_id|user_id|     product|       category|amount|          timestamp|
+--------------+-------+------------+---------------+------+-------------------+
|             1|    101|      Laptop|    Electronics| 55000|2025-02-01 10:15:00|
|             2|    102|       Phone|    Electronics| 25000|2025-02-01 11:20:00|
|             3|    103|       Shirt|        Fashion|  1500|2025-02-02 09:10:00|
|             4|    104|       Shoes|        Fashion|  3200|2025-02-02 12:45:00|
|             5|    105|       Mixer|Home Appliances|  4800|2025-02-03 14:30:00|
|             6|    101|  Headphones|    Electronics|  2200|2025-02-03 16:00:00|
|             7|    106|        Book|      Education|   600|2025-02-04 10:05:00|
|             8|    107|       Chair|      Furniture|  7200|2025-02-04 13:50:00|
|             9|    108|       Watch|        Fashion|  5400|2025-02-05 17:25:00|
|            10|    109|    

In [25]:
df.head(3)

[Row(transaction_id=1, user_id=101, product='Laptop', category='Electronics', amount=55000, timestamp=datetime.datetime(2025, 2, 1, 10, 15)),
 Row(transaction_id=2, user_id=102, product='Phone', category='Electronics', amount=25000, timestamp=datetime.datetime(2025, 2, 1, 11, 20)),
 Row(transaction_id=3, user_id=103, product='Shirt', category='Fashion', amount=1500, timestamp=datetime.datetime(2025, 2, 2, 9, 10))]

In [24]:
df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [27]:
df.filter(df.amount >  50000).show()

+--------------+-------+-------------+-----------+------+-------------------+
|transaction_id|user_id|      product|   category|amount|          timestamp|
+--------------+-------+-------------+-----------+------+-------------------+
|             1|    101|       Laptop|Electronics| 55000|2025-02-01 10:15:00|
|            45|    144|Graphics Card|Electronics| 55000|2025-02-13 17:30:00|
|            65|    164|        Drone|Electronics| 65000|2025-02-17 16:55:00|
|            98|    197|     Smart TV|Electronics| 52000|2025-02-24 13:30:00|
+--------------+-------+-------------+-----------+------+-------------------+



In [34]:
df.groupBy('user_id').count().show(n=100)

+-------+-----+
|user_id|count|
+-------+-----+
|    148|    1|
|    137|    1|
|    133|    1|
|    108|    1|
|    155|    1|
|    193|    1|
|    101|    2|
|    115|    1|
|    126|    1|
|    183|    1|
|    159|    1|
|    192|    1|
|    103|    1|
|    128|    1|
|    209|    1|
|    122|    1|
|    157|    1|
|    190|    1|
|    111|    1|
|    140|    1|
|    177|    1|
|    132|    1|
|    152|    1|
|    185|    1|
|    146|    1|
|    206|    1|
|    182|    1|
|    168|    1|
|    205|    1|
|    142|    1|
|    178|    1|
|    164|    1|
|    169|    1|
|    139|    1|
|    120|    1|
|    163|    1|
|    191|    1|
|    117|    1|
|    154|    1|
|    112|    1|
|    165|    1|
|    179|    1|
|    189|    1|
|    207|    1|
|    127|    1|
|    197|    1|
|    107|    1|
|    202|    1|
|    175|    1|
|    196|    1|
|    114|    1|
|    173|    1|
|    161|    1|
|    176|    1|
|    162|    1|
|    130|    1|
|    136|    1|
|    171|    1|
|    194|    1|
|    129

In [40]:
df.createOrReplaceTempView('tran')

In [44]:
spark.sql("""
SELECT category, count(*) as count FROM tran WHERE amount > 5000 GROUP BY category
""").show()

+---------------+-----+
|       category|count|
+---------------+-----+
|        Fashion|    2|
|    Electronics|   18|
|Home Appliances|    9|
|      Furniture|   14|
+---------------+-----+



# RDD

In [ ]:
sc = spark.sparkContext

In [4]:
rdd = sc.textFile('transactions.csv')

In [5]:
header = rdd.first()

In [6]:
data = rdd.filter(lambda x: x != header)

In [7]:
data.collect()

['1,101,Laptop,Electronics,55000,2025-02-01 10:15:00',
 '2,102,Phone,Electronics,25000,2025-02-01 11:20:00',
 '3,103,Shirt,Fashion,1500,2025-02-02 09:10:00',
 '4,104,Shoes,Fashion,3200,2025-02-02 12:45:00',
 '5,105,Mixer,Home Appliances,4800,2025-02-03 14:30:00',
 '6,101,Headphones,Electronics,2200,2025-02-03 16:00:00',
 '7,106,Book,Education,600,2025-02-04 10:05:00',
 '8,107,Chair,Furniture,7200,2025-02-04 13:50:00',
 '9,108,Watch,Fashion,5400,2025-02-05 17:25:00',
 '10,109,Tablet,Electronics,30000,2025-02-06 18:40:00',
 '11,110,Camera,Electronics,42000,2025-02-07 09:15:00',
 '12,111,Backpack,Fashion,1800,2025-02-07 10:40:00',
 '13,112,Refrigerator,Home Appliances,38000,2025-02-07 12:10:00',
 '14,113,Notebook,Education,120,2025-02-07 14:05:00',
 '15,114,Sofa,Furniture,25000,2025-02-07 16:20:00',
 '16,115,Smartwatch,Electronics,15000,2025-02-08 09:30:00',
 '17,116,Jeans,Fashion,2200,2025-02-08 11:00:00',
 '18,117,Microwave,Home Appliances,8500,2025-02-08 13:25:00',
 '19,118,Desk,Furnit

In [8]:
parsed = data.map( lambda x : x.split(',')).map(lambda x : (int(x[0]), int(x[1]),x[2], x[3], int(x[4]), x[5] ))

In [13]:
parsed.collect()

[(1, 101, 'Laptop', 'Electronics', 55000, '2025-02-01 10:15:00'),
 (2, 102, 'Phone', 'Electronics', 25000, '2025-02-01 11:20:00'),
 (3, 103, 'Shirt', 'Fashion', 1500, '2025-02-02 09:10:00'),
 (4, 104, 'Shoes', 'Fashion', 3200, '2025-02-02 12:45:00'),
 (5, 105, 'Mixer', 'Home Appliances', 4800, '2025-02-03 14:30:00'),
 (6, 101, 'Headphones', 'Electronics', 2200, '2025-02-03 16:00:00'),
 (7, 106, 'Book', 'Education', 600, '2025-02-04 10:05:00'),
 (8, 107, 'Chair', 'Furniture', 7200, '2025-02-04 13:50:00'),
 (9, 108, 'Watch', 'Fashion', 5400, '2025-02-05 17:25:00'),
 (10, 109, 'Tablet', 'Electronics', 30000, '2025-02-06 18:40:00'),
 (11, 110, 'Camera', 'Electronics', 42000, '2025-02-07 09:15:00'),
 (12, 111, 'Backpack', 'Fashion', 1800, '2025-02-07 10:40:00'),
 (13, 112, 'Refrigerator', 'Home Appliances', 38000, '2025-02-07 12:10:00'),
 (14, 113, 'Notebook', 'Education', 120, '2025-02-07 14:05:00'),
 (15, 114, 'Sofa', 'Furniture', 25000, '2025-02-07 16:20:00'),
 (16, 115, 'Smartwatch', 'E

In [24]:
cs = parsed.map(lambda x : (x[3], x[4])).reduceByKey(lambda a, b: a+b)

In [25]:
cs.collect()

[('Home Appliances', 206700),
 ('Education', 8270),
 ('Furniture', 225000),
 ('Electronics', 617600),
 ('Fashion', 58300)]

In [ ]:
totalrev = parsed.map(lambda x : x[4]).reduce(lambda a,b: a+b)

In [27]:
totalrev

1115870

In [57]:
ta1 = parsed.filter(lambda x : x[4] > 50000).map(lambda x : 1).reduce(lambda a ,b: a+b)

In [58]:
ta1

4

In [55]:
ta1 = parsed.filter(lambda x: x[4] > 50000) \
            .map(lambda x: 1) \
            .reduce(lambda a, b: a + b)

#  Machine Learning

In [59]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MLExample") \
    .getOrCreate()

26/02/26 15:52:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [74]:
data = [
    (25, 50000, 0),
    (35, 80000, 1),
    (45, 90000, 1),
    (23, 40000, 0)
]

df = spark.createDataFrame(data, ["age", "income", "label"])
df.show()

+---+------+-----+
|age|income|label|
+---+------+-----+
| 25| 50000|    0|
| 35| 80000|    1|
| 45| 90000|    1|
| 23| 40000|    0|
+---+------+-----+



In [75]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['age','income'], outputCol='features')

In [67]:
df = assembler.transform(df)

In [68]:
df.select('features', 'label').show()

+--------------+-----+
|      features|label|
+--------------+-----+
|[25.0,50000.0]|    0|
|[35.0,80000.0]|    1|
|[45.0,90000.0]|    1|
|[23.0,40000.0]|    0|
+--------------+-----+



In [69]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='label')

In [70]:
model = lr.fit(df)

In [71]:
predictions = model.transform(df)
predictions.select("features", "prediction").show()

+--------------+----------+
|      features|prediction|
+--------------+----------+
|[25.0,50000.0]|       0.0|
|[35.0,80000.0]|       1.0|
|[45.0,90000.0]|       1.0|
|[23.0,40000.0]|       0.0|
+--------------+----------+



In [76]:
from pyspark.ml import Pipeline

pl = Pipeline(stages=[assembler,lr])

In [77]:
model = pl.fit(df)

In [78]:
predictions = model.transform(df)
predictions.show()

+---+------+-----+--------------+--------------------+--------------------+----------+
|age|income|label|      features|       rawPrediction|         probability|prediction|
+---+------+-----+--------------+--------------------+--------------------+----------+
| 25| 50000|    0|[25.0,50000.0]|[17.7675669081009...|[0.99999998078485...|       0.0|
| 35| 80000|    1|[35.0,80000.0]|[-17.735785205939...|[1.98356457826680...|       1.0|
| 45| 90000|    1|[45.0,90000.0]|[-37.609059292318...|[4.64080242904261...|       1.0|
| 23| 40000|    0|[23.0,40000.0]|[27.9942529364414...|[0.99999999999930...|       0.0|
+---+------+-----+--------------+--------------------+--------------------+----------+



# Spark Streaming

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StreamingExample") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/27 07:36:35 WARN Utils: Your hostname, aswin-Inspiron, resolves to a loopback address: 127.0.1.1; using 192.168.127.189 instead (on interface wlp1s0)
26/02/27 07:36:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 07:36:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
stream_df = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

26/02/27 07:37:05 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [4]:
from pyspark.sql.functions import explode, split

words = stream_df.select(
    explode(split(stream_df.value, " ")).alias("word")
)

word_counts = words.groupBy("word").count()

In [5]:
query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

26/02/27 07:37:39 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-9caa3062-a010-43e2-9ddf-24fadd9f7865. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/02/27 07:37:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/02/27 07:37:39 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/02/27 07:37:41 ERROR MicroBatchExecution: Query [id = b5e240b5-6023-4b73-a6e4-e3fd5022b47a, runId = de932b75-dee4-419b-bec8-3639d334cd64] terminated with error
java.net.ConnectException: Connection refused
	at java.base/sun.nio.ch.Net.connect0(Native Method)
	at java.base/sun.nio.ch.Net.connect(Net.java:591)
	at java.base/sun.nio.ch.Net.connect(Net.java:580)

StreamingQueryException: [STREAM_FAILED] Query [id = b5e240b5-6023-4b73-a6e4-e3fd5022b47a, runId = de932b75-dee4-419b-bec8-3639d334cd64] terminated with exception: Connection refused SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = b5e240b5-6023-4b73-a6e4-e3fd5022b47a, runId = de932b75-dee4-419b-bec8-3639d334cd64]
Current Committed Offsets: {}
Current Available Offsets: {TextSocketV2[host: localhost, port: 9999]: -1}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSource org.apache.spark.sql.execution.streaming.ConsoleTable$@6cd34df5, b5e240b5-6023-4b73-a6e4-e3fd5022b47a, Complete
+- ~Aggregate [word#3], [word#3, count(1) AS count#4L]
   +- ~Project [word#3]
      +- ~Generate explode(split(value#0,  , -1)), false, [word#3]
         +- ~StreamingDataSourceV2ScanRelation[value#0] Socket[localhost:9999]


In [6]:
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName('sparapp').getOrCreate()

26/02/27 09:24:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [66]:
df = spark.read.csv('c1.csv',header=True,inferSchema=True)

In [67]:
df.dropna()

DataFrame[customerID: string, gender: string, SeniorCitizen: int, Partner: string, Dependents: string, tenure: int, PhoneService: string, MultipleLines: string, InternetService: string, OnlineSecurity: string, OnlineBackup: string, DeviceProtection: string, TechSupport: string, StreamingTV: string, StreamingMovies: string, Contract: string, PaperlessBilling: string, PaymentMethod: string, MonthlyCharges: double, TotalCharges: double, Churn: string]

In [68]:
df.printSchema(())

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [39]:
from pyspark.sql.functions import col

df = df.withColumn('TotalCharges', col('TotalCharges').cast('double'))

In [53]:
df = df.withColumn('MonthlyCharges', col('MonthlyCharges').cast('double'))
df = df.withColumn('tenure', col('tenure').cast('double'))
df = df.withColumn('SeniorCitizen', col('SeniorCitizen').cast('double'))



In [54]:
df.show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|     OnlineSecurity|       OnlineBackup|   DeviceProtection|        TechSupport|        StreamingTV|    StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|          0.0|    Yes|        No|   1.0|  

In [55]:
df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: double (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: double (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [69]:
ct = [
    "gender", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]

In [70]:
from pyspark.ml.feature import StringIndexer


sti = [StringIndexer(inputCol=col, outputCol= col+'i') for col in ct]

In [71]:
li = StringIndexer(inputCol='Churn', outputCol='label')

In [72]:
nc = numeric_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [73]:
feat = nc + [col+'i' for col in ct]

In [74]:
from pyspark.ml.feature import VectorAssembler

vs = VectorAssembler(inputCols= feat, outputCol='features')

In [75]:
from pyspark.ml.classification import LogisticRegression


lr = LogisticRegression(featuresCol='features', labelCol='label')



In [76]:
from pyspark.ml import Pipeline

pipe = Pipeline(stages=sti +[li, vs, lr])

In [77]:
df = df.dropna(subset=[
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
])

In [78]:
model = pipe.fit(df)

In [81]:
df.createOrReplaceTempView('table')

In [87]:
s = spark.sql("""
select * from table where Partner = "Yes" and PhoneService = 'No' and gender = 'Female' and SeniorCitizen = 1 """).show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|5386-THSLQ|Female|            1|    Yes|        No|    66|          No|No phone service|            DSL|            No|         Yes|             Yes|         No|    

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number
wi = Window.partitionBy('PaymentMethod').orderBy('TotalCharges')

In [92]:
df.withColumn("row_num", row_number().over(wi))

NameError: name 'row_number' is not defined